In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [20]:
import os
from glob import glob
import pandas as pd
import numpy as np
from tqdm import tqdm
from itertools import combinations
from multiprocessing import Pool

In [4]:
from slidingwinalignment.io import read_fasta, save_pickle, load_pickle
from slidingwinalignment.alignment import transform_similarity_matrix, local_alignment
from slidingwinalignment.matrices import similarity_matrix
from slidingwinalignment.utils import pdb_root
from slidingwinalignment.metrics import get_similarity_score, get_similarity_score_local

## configuration

Please adjust the number of CPU workers based on your machine.

Set `N_CPU = 1` to run the screening step without multiprocessing.

In [19]:
# User-configurable parameters
N_CPU = 4

## read sequences from fasta

In [5]:
# read pickle file
path = "data/SH3.fasta"
all_seqs = read_fasta(path)

Read 10 sequences.
Example:
2J6K_0
MVDYIVEYDYDAVHDDELTIRVGEIIRNVKKLQEEGWLEGELNGRRGMFPDNFVKEIKRETEFKDDSLPIKRERHGNVASLVQRISTYGLPAGGIQPHPQTKNIKKKTKKRQCKVLFEYIPQNEDELELKVGDIIDINEEVEEGWWSGTLNNKLGLFPSNFVKELEVTDDGETHEAQDDSETVLAGPTSPIPSLGNVSETASGSVTQPKKIRGIGFGDIFKEGSVKLRTRTSSSETEEKKPEKPLILQSLGPKTQSVEITKTDTEGKIKAKEYCRTLFAYEGTNEDELTFKEGEIIHLISKETGEAGWWRGELNGKEGVFPDNFAVQINELDKDFPKPKKPPPPAKAPAPKPELIAAEKKYFSLKPEEKDEKSTLEQKPSKPAAPQVPPKKPTPPTKASNLLRSSGTVYPKRPEKPVPPPPPIAKINGEVSSISSKFETEPVSKLKLDSEQLPLRPKSVDFDSLTVRTSKETDVVNFDDIASSENLLHLTANRPKMPGRRLPGRFNGGHSPTHSPEKILKLPKEEDSANLKPSELKKDTCYSPKPSVYLSTPSSASKANTTAFLTPLEIKAKVETDDVKKNSLDELRAQIIELLCIVEALKKDHGKELEKLRKDLEEEKTMRSNLEMEIEKLKKAVLSS


## load model and get embedding(using prostt5 as an example here)

In [6]:
from transformers import T5Tokenizer, T5EncoderModel
import torch
import re

In [9]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
tokenizer = T5Tokenizer.from_pretrained('Rostlab/ProstT5', do_lower_case=False)
model = T5EncoderModel.from_pretrained("Rostlab/ProstT5").to(device)
model = model.float() if device.type=='cpu' else model.half()
print("Model loaded.")

Using device: cuda:0


Loading weights: 100%|██████████| 195/195 [00:00<00:00, 60548.51it/s]


Model loaded.


In [10]:
# key and value to two separate lists
keys = list(all_seqs.keys())
values = list(all_seqs.values())

In [11]:
seq_embeddings = []
for i in tqdm(values):
    sequences = [i]
    length = len(sequences[0])
    sequences = [" ".join(list(re.sub(r"[UZOB]", "X", sequence))) for sequence in sequences]
    sequences = [ "<AA2fold>" + " " + s if s.isupper() else "<fold2AA>" + " " + s # this expects 3Di sequences to be already lower-case
                      for s in sequences
                    ]
    ids = tokenizer(sequences,
                    add_special_tokens=True,
                    padding="longest",
                    return_tensors='pt').to(device)
    with torch.no_grad():
      embedding_repr = model(
              ids.input_ids, 
              attention_mask=ids.attention_mask
              )
    emb = embedding_repr.last_hidden_state[0, 1 : length + 1].cpu().numpy()
    seq_embeddings.append(emb)

100%|██████████| 10/10 [00:02<00:00,  4.54it/s]


In [12]:
len(seq_embeddings), len(seq_embeddings[0]), seq_embeddings[0].shape

(10, 639, (639, 1024))

## get matrices

In [13]:
window_size = 3

In [15]:
# for all combinations of sequences, compute the similarity matrix
matrix_dict = {}
for i, j in tqdm(combinations(range(len(seq_embeddings)), 2)):
    key1, key2 = keys[i], keys[j]
    emb1, emb2 = seq_embeddings[i], seq_embeddings[j]
    matrix = similarity_matrix(emb1, emb2, window_size)
    matrix_dict[(key1, key2)] = matrix
list(matrix_dict.items())[0]

45it [00:02, 18.35it/s]


(('2J6K_0', '1E6G_1'),
 array([[0.45389881, 0.37115541, 0.32462353, ..., 0.08594038, 0.08122378,
         0.08366413],
        [0.2826031 , 0.27636317, 0.23794939, ..., 0.11061744, 0.10680729,
         0.09953065],
        [0.2509892 , 0.23576278, 0.21842637, ..., 0.11061622, 0.1102363 ,
         0.10299553],
        ...,
        [0.16326241, 0.1560147 , 0.15048859, ..., 0.58361987, 0.60619457,
         0.54204585],
        [0.16610148, 0.17499794, 0.15876903, ..., 0.58786963, 0.63276002,
         0.59819458],
        [0.18001077, 0.20417873, 0.19191004, ..., 0.59621003, 0.6364786 ,
         0.65114616]], shape=(637, 2475)))

## get regions

In [21]:
_matrix = None
_window = None

def _init(matrix_dict, window_size):
    global _matrix, _window
    _matrix = matrix_dict
    _window = window_size

def _worker(pairname):
    matrix = _matrix[pairname]
    if matrix is None:
        return pairname, pairname[0], pairname[1], None, None, None

    window_size = _window
    matrix = transform_similarity_matrix(matrix, midpoint=0.2, sharpness=9, scale=3)
    score, protein1, protein2 = local_alignment(matrix, 10)

    protein1_residues = (int(protein1[0]), int(protein1[1]) + window_size)
    protein2_residues = (int(protein2[0]), int(protein2[1]) + window_size)

    return pairname, pairname[0], pairname[1], protein1_residues, protein2_residues, score

items = list(matrix_dict.keys())

with Pool(processes=N_CPU, initializer=_init, initargs=(matrix_dict, window_size)) as pool:
    it = pool.imap_unordered(_worker, items, chunksize=1)
    rows = list(tqdm(it, total=len(items)))

results_df = pd.DataFrame(
    rows,
    columns=[
        "pairname",
        "prot1",
        "prot2",
        "protein1_residues",
        "protein2_residues",
        "alignment_score",
    ],
)

100%|██████████| 45/45 [00:14<00:00,  3.10it/s]


In [29]:
results_df = results_df.sort_values("alignment_score", ascending=False)
results_df.head()

,pairname,prot1,prot2,protein1_residues,protein2_residues,alignment_score
36,"(1ABQ_5, 1KIK_6)",1ABQ_5,1KIK_6,"(38, 503)","(38, 504)",1188.111328
43,"(1ABQ_5, 1ZUK_8)",1ABQ_5,1ZUK_8,"(528, 1019)","(366, 857)",1035.798637
6,"(2J6K_0, 1ABQ_5)",2J6K_0,1ABQ_5,"(360, 583)","(795, 1018)",474.060161
7,"(2J6K_0, 1ZUK_8)",2J6K_0,1ZUK_8,"(337, 566)","(506, 727)",419.214163
2,"(2J6K_0, 5QU3_2)",2J6K_0,5QU3_2,"(0, 177)","(2, 175)",372.326348
